# MalApp 推理速度、RAG 与学习闭环诊断

## tl;dr

本笔记本从当前工程数据库与实现文件中提取事实，用于回答推理速度、DPO 数据、Agent 闭环、RAG embedding 和误判学习问题。

## Context & Methods

### Key Assumptions

- 当前工程目录的 `data/mvp.db` 是开发库。
- `release` 中按目录名排序的最新版本数据库用于补充检查，但不假定它就是用户此刻正在运行的进程数据目录。
- 只有 provider 为 `openai_compatible` 的记录计入真实远端大模型延迟。

In [1]:
from pathlib import Path
import collections, json, sqlite3, statistics

ROOT = Path.cwd()
DEV_DB = ROOT / 'data' / 'mvp.db'
release_dbs = sorted((ROOT / 'release').glob('MalApp_AgentTrace_LearningLoop_*/_internal/data/mvp.db'))
LATEST_RELEASE_DB = release_dbs[-1] if release_dbs else None
print('开发库:', DEV_DB)
print('最新可见发布库:', LATEST_RELEASE_DB)


开发库: C:\Users\啤酒肚\Desktop\工作\test1\data\mvp.db
最新可见发布库: C:\Users\啤酒肚\Desktop\工作\test1\release\MalApp_AgentTrace_LearningLoop_20260726_low_failure\_internal\data\mvp.db


## Data

### 1. 统计报告、人工复核、奖励和延迟

In [2]:
def table_names(conn):
    return {row[0] for row in conn.execute("SELECT name FROM sqlite_master WHERE type='table'")}

def percentile(values, p):
    if not values:
        return None
    values = sorted(values)
    return values[round((len(values) - 1) * p)]

def inspect_db(path):
    if path is None or not path.exists():
        return {'path': str(path), 'available': False}
    conn = sqlite3.connect(path)
    tables = table_names(conn)
    counts = {}
    for name in ['judgements', 'agent_traces', 'human_reviews', 'reward_records', 'sample_tasks']:
        counts[name] = conn.execute(f'SELECT COUNT(*) FROM {name}').fetchone()[0] if name in tables else 0
    reports = []
    if 'judgements' in tables:
        reports = [json.loads(row[0]) for row in conn.execute('SELECT payload_json FROM judgements ORDER BY created_at DESC LIMIT 1000')]
    groups = collections.defaultdict(list)
    token_rows = []
    for report in reports:
        debate = report.get('debate') or {}
        providers = debate.get('providers') or {}
        def backend(name):
            item = providers.get(name)
            return item.get('backend') if isinstance(item, dict) else item
        key = (debate.get('execution_mode') or '', backend('model_a'), backend('model_b'))
        metrics = debate.get('metrics') or {}
        latency = metrics.get('latency_ms')
        if isinstance(latency, (int, float)) and latency > 0:
            groups[key].append(float(latency))
        usage = metrics.get('token_usage') or {}
        if usage:
            token_rows.append(usage)
    group_summary = []
    for key, values in groups.items():
        group_summary.append({
            'execution_mode': key[0], 'model_a_backend': key[1], 'model_b_backend': key[2],
            'n': len(values), 'p50_ms': round(statistics.median(values)),
            'p90_ms': round(percentile(values, .9)), 'max_ms': round(max(values)),
        })
    conn.close()
    return {'path': str(path), 'available': True, 'counts': counts, 'latency_groups': group_summary, 'token_rows': token_rows[:10]}

db_diagnostics = [inspect_db(DEV_DB), inspect_db(LATEST_RELEASE_DB)]
print(json.dumps(db_diagnostics, ensure_ascii=False, indent=2))


[
  {
    "path": "C:\\Users\\啤酒肚\\Desktop\\工作\\test1\\data\\mvp.db",
    "available": true,
    "counts": {
      "judgements": 269,
      "agent_traces": 14,
      "human_reviews": 0,
      "reward_records": 14,
      "sample_tasks": 8217
    },
    "latency_groups": [
      {
        "execution_mode": "full_debate",
        "model_a_backend": "rule",
        "model_b_backend": "rule",
        "n": 97,
        "p50_ms": 6,
        "p90_ms": 33,
        "max_ms": 72
      },
      {
        "execution_mode": "full_debate",
        "model_a_backend": "openai_compatible",
        "model_b_backend": "openai_compatible",
        "n": 2,
        "p50_ms": 187950,
        "p90_ms": 189578,
        "max_ms": 189578
      },
      {
        "execution_mode": "",
        "model_a_backend": "rule",
        "model_b_backend": "rule",
        "n": 133,
        "p50_ms": 2,
        "p90_ms": 3,
        "max_ms": 8
      }
    ],
    "token_rows": [
      {
        "prompt_tokens": 26076,
        "

### 2. 核查 RAG 向量维度与 embedding 配置

In [3]:
rag_db = ROOT / 'data' / 'rag' / 'rag_store.db'
rag_stats = {'path': str(rag_db), 'documents': 0, 'dimensions': {}}
if rag_db.exists():
    conn = sqlite3.connect(rag_db)
    rows = conn.execute('SELECT embedding_json FROM rag_documents').fetchall()
    dimensions = collections.Counter(len(json.loads(row[0])) for row in rows)
    rag_stats = {'path': str(rag_db), 'documents': len(rows), 'dimensions': dict(dimensions)}
    conn.close()
embedding_source = (ROOT / 'engine' / 'rag' / 'embedding.py').read_text(encoding='utf-8')
embedding_facts = {
    'configured_default_model': 'BAAI/bge-small-zh-v1.5' in embedding_source,
    'local_files_only_default': 'MALAPP_RAG_EMBED_LOCAL_FILES_ONLY' in embedding_source,
    'hash_fallback_present': 'local_hash_embedding' in embedding_source,
    'rag_stats': rag_stats,
}
print(json.dumps(embedding_facts, ensure_ascii=False, indent=2))


{
  "configured_default_model": true,
  "local_files_only_default": true,
  "hash_fallback_present": true,
  "rag_stats": {
    "path": "C:\\Users\\啤酒肚\\Desktop\\工作\\test1\\data\\rag\\rag_store.db",
    "documents": 376,
    "dimensions": {
      "384": 376
    }
  }
}


## Results

### 3. 当前闭环能力检查

In [4]:
files = {
    'trace_and_review': ROOT / 'engine' / 'agent_trace.py',
    'reward': ROOT / 'engine' / 'reward_builder.py',
    'dataset_export': ROOT / 'engine' / 'dataset_export.py',
    'debate': ROOT / 'engine' / 'debate_flow.py',
    'pipeline': ROOT / 'engine' / 'pipeline.py',
}
texts = {name: path.read_text(encoding='utf-8') for name, path in files.items()}
capabilities = {
    'agent_trace_saved': 'save_agent_trace' in texts['trace_and_review'],
    'human_review_saved': 'save_human_review' in texts['trace_and_review'],
    'reward_record_saved': 'save_reward_for_report' in texts['reward'],
    'dpo_export_exists': 'build_dpo_rows' in texts['dataset_export'],
    'policy_export_exists': 'build_policy_rows' in texts['dataset_export'],
    'full_debate_forced': 'debate_config["verification_mode"] = False' in texts['pipeline'],
    'parallel_model_calls': 'ThreadPoolExecutor' in texts['debate'],
    'format_failure_log': 'llm_validation_failures.jsonl' in texts['debate'],
}
print(json.dumps(capabilities, ensure_ascii=False, indent=2))


{
  "agent_trace_saved": true,
  "human_review_saved": true,
  "reward_record_saved": true,
  "dpo_export_exists": true,
  "policy_export_exists": true,
  "full_debate_forced": true,
  "parallel_model_calls": true,
  "format_failure_log": true
}


In [5]:
assert db_diagnostics[0]['available']
assert embedding_facts['rag_stats']['documents'] >= 0
assert capabilities['agent_trace_saved'] and capabilities['human_review_saved']
assert capabilities['reward_record_saved'] and capabilities['dpo_export_exists']
print('校验通过：数据库、RAG 配置和闭环实现检查均完成。')


校验通过：数据库、RAG 配置和闭环实现检查均完成。


## Takeaways

- 当前系统已经具备轨迹、人工复核、reward 和 DPO/policy 数据导出骨架，但人工复核数据量决定闭环是否真正运转。
- 默认 embedding 目标是 BAAI/bge-small-zh-v1.5；模型文件不可用时会退化为 384 维本地哈希向量，因此“384 维”本身不能证明正在使用 BGE。
- 流水线强制每个未命中缓存的样本进入完整辩论；这是速度优化的首要结构性杠杆。